In [ ]:
path_prefix = path_prefix = "/Users/agobharoun/Documents/IND320/Streamlit_dashboard/streamlit/"

# IND320 Project Work – Part 1

In this notebook, I will prepare and explore the Norwegian reservoir
dataset before using the same data in my Streamlit app.

## 1. Reading the reservoir data

The CSV file is stored in the `data` folder, while this notebook is stored
in `notebooks`. I use a relative path instead of the full path on my
computer so the notebook can still find the data when the repository is
downloaded or cloned by someone else.

In [11]:
import pandas as pd

# The two dots take me one folder back from notebooks/.
# From there, I enter the data folder and read the CSV file.
reservoirs = pd.read_csv("../data/reservoirs.csv")

# The complete dataset is too large to print in the notebook.
# I therefore show its size and the first five rows to check that it loaded correctly.
print(f"The dataset contains {reservoirs.shape[0]:,} rows and {reservoirs.shape[1]} columns.")

reservoirs.head()

The dataset contains 14,877 rows and 11 columns.


,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


In [3]:
print(reservoirs.columns.tolist())

# Display the number of rows and columns.
print("Dataset dimensions:", reservoirs.shape)

# Show each column's data type and number of non-missing values.
reservoirs.info()

['dato_Id', 'omrType', 'omrnr', 'iso_aar', 'iso_uke', 'fyllingsgrad', 'kapasitet_TWh', 'fylling_TWh', 'neste_Publiseringsdato', 'fyllingsgrad_forrige_uke', 'endring_fyllingsgrad']
Dataset dimensions: (14877, 11)
<class 'pandas.DataFrame'>
RangeIndex: 14877 entries, 0 to 14876
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dato_Id                   14877 non-null  str    
 1   omrType                   14877 non-null  str    
 2   omrnr                     14877 non-null  int64  
 3   iso_aar                   14877 non-null  int64  
 4   iso_uke                   14877 non-null  int64  
 5   fyllingsgrad              14877 non-null  float64
 6   kapasitet_TWh             14877 non-null  float64
 7   fylling_TWh               14877 non-null  float64
 8   neste_Publiseringsdato    14877 non-null  str    
 9   fyllingsgrad_forrige_uke  14877 non-null  float64
 10  endring_fyllingsgrad     

## 2. Preparing the date columns

When I inspected the imported data, both date columns were stored as text.
I convert them to datetime values because this will make it easier to sort,
filter and plot the observations later.

The next-publication column contains the date `0001-01-01` many times.
This is not a realistic publication date, so I interpret it as a
placeholder for unavailable information and replace it with a missing value.

In [12]:
# The observation dates were imported as text. I convert them to datetime
# so Pandas can treat them as chronological values instead of ordinary strings.
reservoirs["dato_Id"] = pd.to_datetime(
    reservoirs["dato_Id"],
    format="%Y-%m-%d"
)

# I count the placeholder dates before replacing them so I can document
# how many values were affected by this cleaning decision.
placeholder_date = "0001-01-01T00:00:00"
placeholder_count = (
    reservoirs["neste_Publiseringsdato"] == placeholder_date
).sum()

# Year 0001 is used as a placeholder in the source data.
# I replace it with a proper missing value before converting the column.
reservoirs["neste_Publiseringsdato"] = reservoirs[
    "neste_Publiseringsdato"
].replace(placeholder_date, pd.NA)

# I use errors="coerce" as a safeguard. If another value cannot be
# interpreted as a date, Pandas will record it as missing instead of stopping.
reservoirs["neste_Publiseringsdato"] = pd.to_datetime(
    reservoirs["neste_Publiseringsdato"],
    format="ISO8601",
    errors="coerce"
)

# I format the dates in the Norwegian day.month.year style for this summary.
first_date = reservoirs["dato_Id"].min().strftime("%d.%m.%Y")
last_date = reservoirs["dato_Id"].max().strftime("%d.%m.%Y")

print(f"The observations cover the period from {first_date} to {last_date}.")
print(f"I replaced {placeholder_count:,} placeholder publication dates.")

The observations cover the period from 08.01.1995 to 06.09.2026.
I replaced 11,322 placeholder publication dates.


In [13]:
placeholder_percentage = placeholder_count / len(reservoirs) * 100

print(
    f"I marked {placeholder_count:,} publication dates as missing "
    f"({placeholder_percentage:.1f}% of the rows)."
)

I marked 11,322 publication dates as missing (76.1% of the rows).


The placeholder occurs in 11,322 rows, corresponding to 76.1% of the
dataset. All placeholders belong to observations before February 2019,
while later observations contain actual publication dates. This suggests
that the next-publication field was added to the source at a later stage.
The affected rows are kept because their reservoir measurements are still
valid; only the unavailable publication date is marked as missing.